%cd /kaggle/working/
import os
import shutil

folder = "/kaggle/working"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)      # delete file or link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # delete folder
    except Exception as e:
        print(f"Failed to delete {file_path}. Reason: {e}")

print("✅ /kaggle/working has been cleaned.")


In [2]:
# --- CELL 1: SETUP & START SERVER ---
import os
import subprocess
import time
import sys

# CẤU HÌNH CƠ BẢN
GIT_REPO_URL = "https://github.com/trungkiet2005/TriAd_Project.git"
PROJECT_DIR_NAME = "TriAd_Project"
SUB_PROJECT_DIR = ""



def run_command(command, cwd=None, env=None):
    print(f"Running: {command}")
    process = subprocess.Popen(command, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None: break
        if output: print(output.strip())
    if process.poll() != 0: print(process.stderr.read())

print("=== 1. CLONING & INSTALLING ===")
if not os.path.exists(PROJECT_DIR_NAME):
    run_command(f"git clone {GIT_REPO_URL}")

working_dir = os.path.join(PROJECT_DIR_NAME, SUB_PROJECT_DIR)
run_command(f"pip install -q -r requirements.txt", cwd=working_dir)
run_command("pip install -q vllm openai python-dotenv mistralai striprtf")


=== 1. CLONING & INSTALLING ===
Running: git clone https://github.com/trungkiet2005/TriAd_Project.git
Running: pip install -q -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.9/405.9 kB 10.1 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.9 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 80.3 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.4/536.4 kB 21.8 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 21.8 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.2 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 14.2 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 7.9 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 5.0 MB/s eta 0:00:00
Running: pip install -q vllm openai python-dotenv mistralai striprtf
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.7 MB/s eta 0:00:00
━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import json
CONFIG_FILE_PATH = "/kaggle/working/TriAd_Project/experiment_configs/qwen32b_noise05.json"  # <--- EDIT THIS LINE

with open(CONFIG_FILE_PATH, "r") as file:
    config = json.load(file)

MODEL_NAME = config["MODEL_NAME"]

os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["VLLM_API_KEY"] = "EMPTY"

In [ ]:
print(f"\n=== 2. STARTING VLLM SERVER ({MODEL_NAME}) ===")
log_file = open("vllm_log.txt", "w")
# Start vLLM background process
cmd = f"python -m vllm.entrypoints.openai.api_server --model {MODEL_NAME} --trust-remote-code --port 8000 --gpu-memory-utilization 0.95"
vllm_process = subprocess.Popen(cmd, shell=True, stdout=log_file, stderr=log_file)

print("Waiting for server readiness...")
for i in range(600): # 10 mins max
    try:
        import urllib.request
        with urllib.request.urlopen("http://localhost:8000/v1/models") as response:
            if response.status == 200:
                print("\n✅ vLLM Server is READY!")
                break
    except:
        pass
    if i % 10 == 0: print(".", end="", flush=True)
    time.sleep(1)



=== 2. STARTING VLLM SERVER (Qwen/Qwen2.5-32B-Instruct) ===
Waiting for server readiness...
....

In [ ]:
%%writefile /kaggle/working/TriAd_Project/main.py

"""
Main Experiment Runner

Reads all JSON configuration files from 'experiment_configs' and executes them.
Usage:
    python main.py
"""

import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path(__file__).parent))

from src.io_managers.file_manager import FileManager
from src.noise_fairgame_factory import NoiseFairGameFactory
from src.checkers.time_checker import TimeChecker
from src.checkers.rule_checker import RuleChecker
from src.checkers.aggregation_checker import AggregationChecker

# Load environment variables (API keys)
load_dotenv()

# String 'http://localhost:8000/v1' or 'https://api.openai.com/v1'
# Ensure these are set correctly for your environment
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["VLLM_API_KEY"] = "EMPTY" 

# ==========================================
#        MANUAL CONFIGURATION
# ==========================================
# Set this to the path of the config file you want to run.
# If set, this takes precedence over command line arguments and auto-discovery.
# Example: "experiment_configs/llama70b_noise00.json"


def main():
    print(f"--- Starting Experiment Runner ---")
    
    # Determine which config(s) to run
    config_files = []
    
    # 1. Check Manual Config Variable
    if CONFIG_FILE_PATH:
        path = Path(CONFIG_FILE_PATH)
        if path.exists():
            config_files = [path]
            print(f"Running manual config defined in script: {path.name}")
        else:
            print(f"Error: Manual config file not found: {path} (Check CONFIG_FILE_PATH)")
            return
            
    # 2. Check Command Line Argument (if no manual config set)
    elif len(sys.argv) > 1:
        config_path = Path(sys.argv[1])
        if not config_path.exists():
            print(f"Error: Config file not found: {config_path}")
            return
        config_files = [config_path]
        print(f"Running single config from argument: {config_path.name}")
        
    # 3. Default: Run All Files in experiment_configs
    else:
        # Locate experiment_configs directory
        config_dir = Path(__file__).parent / "experiment_configs"
        if not config_dir.exists():
            print(f"Error: Config directory not found: {config_dir}")
            print("Please run generate_configs.py first.")
            return

        # Find all JSON config files
        config_files = list(config_dir.glob("*.json"))
        if not config_files:
            print(f"No .json files found in {config_dir}")
            return
        
        print(f"No config file specified. Running ALL {len(config_files)} files in {config_dir}")

    # Initialize checkers (these are stateless or reset per game in factory)
    checkers = [TimeChecker(), RuleChecker(), AggregationChecker()]

    for i, config_file in enumerate(config_files):
        print(f"\n[{i+1}/{len(config_files)}] Loading config: {config_file.name}")
        try:
            # Load configuration
            config = FileManager.read_json_file(config_file)
            
            # Extract LLM display name for factory/output purposes
            llm_name = config.get('llmDisplayName', config.get('llm', 'UnknownLLM'))
            
            # Create Factory
            factory = NoiseFairGameFactory(
                checkers=checkers,
                llm_name=llm_name
            )

            print(f"Running games for {llm_name}...")
            # Run the experiment defined in this config
            factory.create_and_run_games(config)
            
            print(f"Completed {config_file.name}")
            
        except Exception as e:
            print(f"Error processing {config_file.name}: {e}")
            import traceback
            traceback.print_exc()

    print("\n--- All Experiments Complete ---")

if __name__ == "__main__":
    main()


In [ ]:
%cd /kaggle/working/TriAd_Project
!export VLLM_BASE_URL="http://localhost:8000/v1"
!export VLLM_API_KEY="EMPTY"

# Chạy script experiment đã config ở trên
!python main.py

In [ ]:
import shutil
import os

src = "/kaggle/working/TriAd_Project/resources/results"
dst = "/kaggle/working/results"

# Copy thư mục
if os.path.exists(dst):
    shutil.rmtree(dst)  # xóa nếu đã tồn tại

shutil.copytree(src, dst)

print("Copy done!")

# Zip thư mục results
zip_path = "/kaggle/working/results.zip"
shutil.make_archive("/kaggle/working/results", 'zip', dst)

print("Zip done! File saved at:", zip_path)